
# NB19 — LDES Portfolio Mapping & Geographic Alignment
**Phase 2 — GB Constraint Analytics Platform**
 
 **Scope:** Map the Ofgem Window 1 LDES minded-to portfolio to historical Phase 1
 constraint groups. Generate technology-aware alignment scores.
 
 **Methodological Corrections applied here:**
 - ✅ **Correction 1 (Topography Trap):** Technology-bifurcated alignment scoring.
 - 🚩 **Correction 3 (RRT Ghost):** `rrt_applicable` flag planted for NB22.
 - 🔄 **Task 1 (NB24 Handoff):** Carries forward ALL overlapping constraint groups 
   per region as a list for NB24 SJR numerator calculation.
 
 **Outputs:** `ldes_portfolio_aligned.parquet`

In [13]:
## 0. Imports and Configuration
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.3f}".format)

# ─── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR = Path("..") if Path("../data").exists() else Path(".")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROCESSED_DIR

LDES_PORTFOLIO_PATH = RAW_DIR / "ofgem_ldes_minded_to.csv"
CONSTRAINT_BURDEN_PATH = PROCESSED_DIR / "constraint_burden_historical.parquet"
OUTPUT_PATH = OUTPUT_DIR / "ldes_portfolio_aligned.parquet"

# ─── Constants ────────────────────────────────────────────────────────────────
TECH_CATEGORY_MAP = {
    "PSH": "PSH",
    "Li-ion": "Li-ion_Flow",
    "Flow Battery": "Li-ion_Flow",
    "LAES": "Li-ion_Flow",
    "CAES": "Li-ion_Flow",
    "Gravity": "Li-ion_Flow",
    "Thermal": "Li-ion_Flow",
    "Other": "Li-ion_Flow",
}
VALID_STATUSES = {"minded_to", "approved"}
DURATION_WARN_BOUNDS = (6.0, 150.0)
RRT_PERSISTENCE_THRESHOLD = 0.40
RRT_HOURS_THRESHOLD = 200.0
DURATION_CAP_HOURS = 18.0
PSH_CONNECTIVITY_CREDIT = 0.05

WEIGHTS_LI_ION_FLOW = {"flexibility_need": 0.50, "electrical_proximity": 0.30, "duration_credit": 0.20}
WEIGHTS_PSH = {"geological_feasibility": 0.50, "electrical_proximity": 0.50}

print("NB19 — LDES Portfolio Mapping & Geographic Alignment")
print("=" * 65)

NB19 — LDES Portfolio Mapping & Geographic Alignment


In [14]:
import pandas as pd
df_real = pd.read_csv(LDES_PORTFOLIO_PATH)
df_real.head()


,project_id,project_name,developer,technology_type,mw_capacity,mwh_capacity,duration_hours,grid_region,bsp_node,latitude,longitude,ofgem_window,status,floor_level_gbp_mw_yr,cap_level_gbp_mw_yr
0,LDES_001,Coire Glas,SSE Renewables,PSH,1440.000,46080.000,32.000,North Scotland,NaN,56.900,-4.900,1,minded_to,NaN,NaN
1,LDES_002,Earba,NaN,PSH,1800.000,27000.000,15.000,North Scotland,NaN,56.800,-4.300,1,minded_to,NaN,NaN
2,LDES_003,Loch Kemp Storage,NaN,PSH,660.000,14718.000,22.300,North Scotland,NaN,57.200,-4.500,1,minded_to,NaN,NaN
3,LDES_004,TeesCAES,NaN,CAES,50.000,1500.000,30.000,North East England,NaN,54.600,-1.200,1,minded_to,NaN,NaN
4,LDES_005,Field Netherton,Field,Li-ion,400.000,6520.000,16.300,North Scotland,NaN,57.300,-2.400,1,minded_to,NaN,NaN


In [15]:
!ls -lh $LDES_PORTFOLIO_PATH
!head $LDES_PORTFOLIO_PATH


-rw-r--r-- 1 ndrew ndrew 1.7K Jun 29 20:28 ../data/raw/ofgem_ldes_minded_to.csv
project_id,project_name,developer,technology_type,mw_capacity,mwh_capacity,duration_hours,grid_region,bsp_node,latitude,longitude,ofgem_window,status,floor_level_gbp_mw_yr,cap_level_gbp_mw_yr
LDES_001,Coire Glas,SSE Renewables,PSH,1440.0,46080.0,32.0,North Scotland,,56.9,-4.9,1,minded_to,,
LDES_002,Earba,,PSH,1800.0,27000.0,15.0,North Scotland,,56.8,-4.3,1,minded_to,,
LDES_003,Loch Kemp Storage,,PSH,660.0,14718.0,22.3,North Scotland,,57.2,-4.5,1,minded_to,,
LDES_004,TeesCAES,,CAES,50.0,1500.0,30.0,North East England,,54.6,-1.2,1,minded_to,,
LDES_005,Field Netherton,Field,Li-ion,400.0,6520.0,16.3,North Scotland,,57.3,-2.4,1,minded_to,,
LDES_006,Field New Deer,Field,Li-ion,400.0,7200.0,18.0,North Scotland,,57.4,-2.2,1,minded_to,,
LDES_007,Field Rigifa,Field,Li-ion,200.0,3600.0,18.0,North Scotland,,57.5,-4.0,1,minded_to,,
LDES_008,Field Fyrish,Field,Li-ion,200.0,3300.0,16.5,North Scotland,,57.7,-4.2,1,minded_t

## 1. Data Ingestion & Validation

In [16]:
def load_and_validate_portfolio(path: Path) -> pd.DataFrame:
    """Load and validate ofgem_ldes_minded_to.csv against the data contract."""
    print(f"\n[1.1] Loading portfolio from {path.name}...")
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip() # Clean any CSV export artifacts
    
    if df["project_id"].duplicated().any():
        raise ValueError("Duplicate project_id values found.")
    for col in ("mw_capacity", "mwh_capacity"):
        if (df[col] <= 0).any():
            raise ValueError(f"{col} must be > 0.")

    df["duration_hours"] = df["mwh_capacity"] / df["mw_capacity"]
    
    lo, hi = DURATION_WARN_BOUNDS
    oob = df[(df["duration_hours"] < lo) | (df["duration_hours"] > hi)]
    if not oob.empty:
        print(f"  ⚠️  Duration warning: {len(oob)} projects outside [{lo}, {hi}]h.")

    df = df[df["status"].isin(VALID_STATUSES)].copy()
    df["technology_category"] = df["technology_type"].map(TECH_CATEGORY_MAP).fillna("Li-ion_Flow")
    
    print(f"  ✅ {len(df)} projects loaded and validated.")
    return df

def load_and_validate_burden(path: Path) -> pd.DataFrame:
    """Load Phase 1 constraint burden parquet and confirm required columns exist."""
    print(f"\n[1.2] Loading Phase 1 constraint burden from {path.name}...")
    df = pd.read_parquet(path)
    required = {"constraint_group", "grid_region", "burden_proxy", "constraint_hours_annual", "constraint_persistence_score"}
    if missing := required - set(df.columns):
        raise ValueError(f"Missing columns: {missing}")
    print(f"  ✅ {len(df)} constraint groups loaded.")
    return df

df_portfolio = load_and_validate_portfolio(LDES_PORTFOLIO_PATH)
df_burden = load_and_validate_burden(CONSTRAINT_BURDEN_PATH)


[1.1] Loading portfolio from ofgem_ldes_minded_to.csv...
  ✅ 16 projects loaded and validated.

[1.2] Loading Phase 1 constraint burden from constraint_burden_historical.parquet...
  ✅ 5 constraint groups loaded.


## 2. Node / Region Matching & NB24 Handoff (Task 1)

In [17]:
def match_constraint_groups(df_portfolio: pd.DataFrame, df_burden: pd.DataFrame) -> pd.DataFrame:
    """Match projects to constraint groups. Carries forward all regional groups for NB24."""
    print("\n[2.1] Matching projects to Phase 1 constraint groups...")

    # 1. Primary Match: Highest burden group per region (Conservative for NB19 scoring)
    region_best = (
        df_burden.sort_values("burden_proxy", ascending=False)
        .groupby("grid_region").first().reset_index()
        [["grid_region", "constraint_group", "burden_proxy", "constraint_hours_annual", "constraint_persistence_score"]]
    )
    
    # 2. TASK 1: ALL matching groups per region (For NB24 SJR Numerator)
    region_all_groups = (
        df_burden.groupby("grid_region")["constraint_group"]
        .apply(list).reset_index()
        .rename(columns={"constraint_group": "all_regional_constraint_groups"})
    )

    df_matched = df_portfolio.merge(region_best, on="grid_region", how="left", suffixes=("", "_burden"))
    df_matched = df_matched.merge(region_all_groups, on="grid_region", how="left")

    df_matched["constraint_group_matched"] = df_matched["constraint_group"]
    df_matched["constraint_group_confidence"] = np.where(df_matched["constraint_group"].notna(), "regional", "inferred")

    # Fallback for inferred matches
    fallback = df_burden.sort_values("burden_proxy").iloc[0]
    mask_inferred = df_matched["constraint_group_confidence"] == "inferred"
    if mask_inferred.any():
        print(f"  ⚠️  {mask_inferred.sum()} inferred matches. Assigning fallback: {fallback['constraint_group']}")
        df_matched.loc[mask_inferred, "constraint_group_matched"] = fallback["constraint_group"]
        df_matched.loc[mask_inferred, "burden_proxy"] = fallback["burden_proxy"]
        df_matched.loc[mask_inferred, "constraint_hours_annual"] = fallback["constraint_hours_annual"]
        df_matched.loc[mask_inferred, "constraint_persistence_score"] = fallback["constraint_persistence_score"]
        # Task 1: Inferred matches only have the fallback group in their list
        df_matched.loc[mask_inferred, "all_regional_constraint_groups"] = df_matched.loc[mask_inferred, "constraint_group_matched"].apply(lambda x: [x])

    df_matched = df_matched.rename(columns={
        "burden_proxy": "burden_proxy_at_location",
        "constraint_hours_annual": "constraint_hours_at_location",
        "constraint_persistence_score": "constraint_persistence_score_at_location",
    }).drop(columns=["constraint_group"], errors="ignore")

    print(f"  Match summary: {df_matched['constraint_group_confidence'].value_counts().to_dict()}")
    return df_matched

df_matched = match_constraint_groups(df_portfolio, df_burden)


[2.1] Matching projects to Phase 1 constraint groups...
  ⚠️  9 inferred matches. Assigning fallback: SSHARN3
  Match summary: {'inferred': 9, 'regional': 7}


## 3. Sub-Score Construction

In [18]:
def build_subscores(df: pd.DataFrame) -> pd.DataFrame:
    """Construct the three normalised sub-scores used in alignment scoring."""
    print("\n[3.1] Building alignment sub-scores...")
    psh_mask = df["technology_category"] == "PSH"

    # Flexibility Need
    bp_min, bp_max = df["burden_proxy_at_location"].min(), df["burden_proxy_at_location"].max()
    df["flexibility_need_score"] = (df["burden_proxy_at_location"] - bp_min) / (bp_max - bp_min) if bp_max > bp_min else 0.5

    # Electrical Proximity
    df["electrical_proximity_score"] = np.where(
        df["constraint_group_confidence"] == "exact", 1.0,
        np.where(df["constraint_group_confidence"] == "regional", 0.85, 0.60)
    )
    df.loc[psh_mask, "electrical_proximity_score"] = (df.loc[psh_mask, "electrical_proximity_score"] + PSH_CONNECTIVITY_CREDIT).clip(upper=1.0)

    # Geological Feasibility (PSH Only)
    if "ofgem_geological_score" in df.columns and df["ofgem_geological_score"].notna().any():
        print("  ✅ Using Ofgem provided geological scores where available.")
        base_geo = df["ofgem_geological_score"]
    else:
        print("  ⚠️  No Ofgem scores. Using latitude/region proxy.")
        base_geo = pd.Series(0.70, index=df.index)
        lat = df["latitude"]
        region = df["grid_region"].fillna("")
        base_geo += np.where(lat >= 57.0, 0.15, np.where(lat >= 56.0, 0.08, np.where(lat >= 55.0, 0.03, 0.0)))
        base_geo += np.where(region.str.contains("North Scotland", na=False), 0.05,
                             np.where(region.str.contains("Wales", na=False), 0.04, 0.0))
        base_geo = base_geo.clip(upper=1.0)

    df["geological_feasibility_score"] = base_geo.where(psh_mask, np.nan).round(3)
    return df

df_matched = build_subscores(df_matched)



[3.1] Building alignment sub-scores...
  ⚠️  No Ofgem scores. Using latitude/region proxy.


## 4. Technology-Aware Alignment Scoring (Correction 1)

In [19]:

def compute_alignment_scores(df: pd.DataFrame) -> pd.DataFrame:
    """Compute raw and adjusted alignment scores using fully vectorized operations."""
    print("\n[4.1] Computing technology-aware alignment scores...")
    psh_mask = df["technology_category"] == "PSH"
    geo = df["geological_feasibility_score"]
    prox = df["electrical_proximity_score"]
    flex = df["flexibility_need_score"]
    dur_credit = np.minimum(df["duration_hours"], DURATION_CAP_HOURS) / DURATION_CAP_HOURS
    
    # Branch B (PSH)
    score_psh_full = WEIGHTS_PSH["geological_feasibility"] * geo + WEIGHTS_PSH["electrical_proximity"] * prox
    score_psh_fallback = (WEIGHTS_LI_ION_FLOW["flexibility_need"] * flex + 
                          WEIGHTS_LI_ION_FLOW["electrical_proximity"] * prox + 
                          WEIGHTS_LI_ION_FLOW["duration_credit"] * dur_credit)
    score_psh = np.where(geo.isna(), score_psh_fallback, score_psh_full)
    
    # Branch A (Li-ion/Flow)
    score_li = (WEIGHTS_LI_ION_FLOW["flexibility_need"] * flex + 
                WEIGHTS_LI_ION_FLOW["electrical_proximity"] * prox + 
                WEIGHTS_LI_ION_FLOW["duration_credit"] * dur_credit)
    
    df["alignment_score_raw"] = np.where(psh_mask, score_psh, score_li)
    df["alignment_score_adjusted"] = df["alignment_score_raw"]
    
    df["notes"] = np.where(
        psh_mask & geo.isna(), "PSH fallback to Li-ion_Flow (geological score unavailable)",
        np.where(psh_mask, "PSH Branch B applied (Correction 1).", "Li-ion_Flow Branch A applied.")
    )
    
    df["alignment_tier"] = pd.cut(df["alignment_score_adjusted"], bins=[-np.inf, 0.40, 0.70, np.inf], labels=["Low", "Medium", "High"])
    return df

df_matched = compute_alignment_scores(df_matched)


[4.1] Computing technology-aware alignment scores...


## 5. RRT Flag Construction (Correction 3)

In [20]:
def flag_rrt_applicable(df: pd.DataFrame) -> pd.DataFrame:
    """Plant the rrt_applicable boolean for NB22. Exempts 'inferred' fallbacks."""
    print("\n[5.1] Setting RRT applicable flag...")
    
    # Condition A requires at least 'regional' confidence to prevent false flags on inferred fallbacks
    cond_a = (
        (df["constraint_persistence_score_at_location"] >= RRT_PERSISTENCE_THRESHOLD) & 
        (df["constraint_group_confidence"].isin(["exact", "regional"]))
    )
    cond_b = (
        (df["constraint_hours_at_location"] >= RRT_HOURS_THRESHOLD) & 
        (df["constraint_group_confidence"].isin(["exact", "regional"]))
    )
    
    df["rrt_applicable"] = cond_a | cond_b
    df["rrt_persistence_score"] = np.where(df["rrt_applicable"], df["constraint_persistence_score_at_location"], np.nan)
    
    print(f"  {df['rrt_applicable'].sum()}/{len(df)} projects flagged rrt_applicable=True")
    print("  ⚠️  RRT penalty NOT applied here. NB22 applies curtailment multiplier.")
    return df

df_matched = flag_rrt_applicable(df_matched)




[5.1] Setting RRT applicable flag...
  7/16 projects flagged rrt_applicable=True
  ⚠️  RRT penalty NOT applied here. NB22 applies curtailment multiplier.


## 6. Output Construction & Export

In [21]:

OUTPUT_COLUMNS = [
    "project_id", "project_name", "developer", "technology_type", "technology_category",
    "mw_capacity", "mwh_capacity", "duration_hours", "grid_region",
    "constraint_group_matched", "constraint_group_confidence", "all_regional_constraint_groups", 
    "burden_proxy_at_location", "constraint_hours_at_location",
    "electrical_proximity_score", "geological_feasibility_score", "flexibility_need_score",
    "alignment_score_raw", "alignment_score_adjusted", "alignment_tier",
    "rrt_applicable", "rrt_persistence_score", "notes",
]

df_output = df_matched[OUTPUT_COLUMNS].copy()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optimized parquet export
df_output.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
print(f"\n[6.1] ✅ Output written → {OUTPUT_PATH}")
print(f"      Rows: {len(df_output)} | Columns: {len(df_output.columns)}")



[6.1] ✅ Output written → ../data/processed/ldes_portfolio_aligned.parquet
      Rows: 16 | Columns: 23


## 7. Diagnostic Summary

In [22]:
print("\n" + "=" * 65)
print("NB19 — DIAGNOSTIC SUMMARY")
print("=" * 65)

print("\n📋 Portfolio Overview:")
summary = df_output.groupby("technology_category").agg(
    n_projects=("project_id", "count"),
    total_mw=("mw_capacity", "sum"),
    total_mwh=("mwh_capacity", "sum"),
    mean_duration_h=("duration_hours", "mean"),
    mean_alignment=("alignment_score_adjusted", "mean"),
).round(2)
print(summary.to_string())

print("\n🎯 Alignment Scores (Correction 1 applied):")
score_view = df_output[[
    "project_id", "technology_type", "technology_category",
    "constraint_group_matched", "alignment_score_adjusted",
    "alignment_tier", "rrt_applicable",
]].sort_values("alignment_score_adjusted", ascending=False)
print(score_view.to_string(index=False))

rrt_flagged = df_output[df_output["rrt_applicable"]]
print(f"\n⚠️  RRT Flag Summary: {len(rrt_flagged)}/{len(df_output)} projects flagged")
print("   (Curtailment penalty deferred to NB22)")

psh_projects = df_output[df_output["technology_category"] == "PSH"]
li_flow_projects = df_output[df_output["technology_category"] == "Li-ion_Flow"]
print(f"\n✅ Correction 1 (Topography Trap) verification:")
print(f"   PSH projects (Branch B — geological scoring): {len(psh_projects)}")
print(f"   Li-ion/Flow projects (Branch A — burden scoring): {len(li_flow_projects)}")

print("\n" + "─" * 65)
print("NB19 complete. Output: ldes_portfolio_aligned.parquet")
print("─" * 65)
print("Next: NB20 — Technology Archetype Economics")


NB19 — DIAGNOSTIC SUMMARY

📋 Portfolio Overview:
                     n_projects  total_mw  total_mwh  mean_duration_h  mean_alignment
technology_category                                                                  
Li-ion_Flow                  13  3745.000  49139.500           13.980           0.500
PSH                           3  3900.000  87798.000           23.100           0.880

🎯 Alignment Scores (Correction 1 applied):
project_id technology_type technology_category constraint_group_matched  alignment_score_adjusted alignment_tier  rrt_applicable
  LDES_007          Li-ion         Li-ion_Flow                   SSEN-S                     0.955           High            True
  LDES_006          Li-ion         Li-ion_Flow                   SSEN-S                     0.955           High            True
  LDES_008          Li-ion         Li-ion_Flow                   SSEN-S                     0.938           High            True
  LDES_005          Li-ion         Li-ion_Flow

Checks

In [23]:
print("🔍 NB19 Input Check:")
print(f"  Portfolio loaded: {len(df_portfolio)} projects")
print(f"  Total MW: {df_portfolio['mw_capacity'].sum():,.0f}")
print(f"  Technologies: {df_portfolio['technology_type'].unique().tolist()}")

🔍 NB19 Input Check:
  Portfolio loaded: 16 projects
  Total MW: 7,645
  Technologies: ['PSH', 'CAES', 'Li-ion', 'Flow Battery']


In [26]:
print("\n🔍 NB19 RRT Flag Check:")

# Load the NB19 output (not the raw input)
df_nb19_output = pd.read_parquet(PROCESSED_DIR / "ldes_portfolio_aligned.parquet")

rrt_true = df_nb19_output[df_nb19_output['rrt_applicable'] == True]
print(f"  RRT-flagged projects: {len(rrt_true)}")
print(f"  Total MW behind constraints: {rrt_true['mw_capacity'].sum():,.0f}")
print(f"  Projects: {rrt_true['project_name'].tolist()}")

# Verify the expected 7 projects
expected_rrt = ['Coire Glas', 'Earba', 'Loch Kemp Storage', 
                'Field Netherton', 'Field New Deer', 'Field Rigifa', 'Field Fyrish']
actual_rrt = sorted(rrt_true['project_name'].tolist())
print(f"\n  ✅ Expected 7 RRT projects: {actual_rrt == sorted(expected_rrt)}")


🔍 NB19 RRT Flag Check:
  RRT-flagged projects: 7
  Total MW behind constraints: 5,100
  Projects: ['Coire Glas', 'Earba', 'Loch Kemp Storage', 'Field Netherton', 'Field New Deer', 'Field Rigifa', 'Field Fyrish']

  ✅ Expected 7 RRT projects: True


In [27]:
print("\n🔍 NB19 Output Check:")
import os
output_path = PROCESSED_DIR / "ldes_portfolio_aligned.parquet"
print(f"  Output exists: {output_path.exists()}")
if output_path.exists():
    df_check = pd.read_parquet(output_path)
    print(f"  Rows: {len(df_check)}")
    print(f"  Has rrt_applicable column: {'rrt_applicable' in df_check.columns}")


🔍 NB19 Output Check:
  Output exists: True
  Rows: 16
  Has rrt_applicable column: True
